In [1]:
import os, json, random, time, copy
from pathlib import Path
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms, models

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"  # mixed precision only on GPU
BATCH_SIZE = 64 if DEVICE.type == "cuda" else 32
NUM_WORKERS = 2 if os.name != "nt" else 0
IMAGE_SIZE = 160
EPOCHS = 2
QUICK_RUN = True
TRAIN_FRACTION = 0.25  # used only when QUICK_RUN=True; set QUICK_RUN=False for all training images

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("Mixed precision:", USE_AMP)
print("Batch size:", BATCH_SIZE)

PyTorch: 2.11.0+cpu
Device: cpu
Mixed precision: False
Batch size: 32


## 2. Set the dataset path

### On Kaggle
1. Open this notebook on Kaggle.
2. Use **Add Input** to attach your dataset.
3. Set `DATA_ROOT` below to the extracted dataset folder under `/kaggle/input/...`.

Common augmented dataset layout:
```text
.../New Plant Diseases Dataset (Augmented)/
    train/<plant___disease>/*.jpg
    valid/<plant___disease>/*.jpg
```

Original PlantVillage layout may instead look like:
```text
PlantVillage/<plant___disease>/*.jpg
```

Edit the path; do not copy the training path from the uploaded ResNet-9 notebook unless your Kaggle input has that exact folder structure.

In [2]:
# CHANGE THIS to your dataset directory.
# Common Kaggle example:
DATA_ROOT = Path("/kaggle/input/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)")
# Local Windows example (uncomment and edit if running on your laptop):
# DATA_ROOT = Path(r"D:\Datasets\PlantVillage")

# For a split dataset, the folder above should contain train/ and valid/ (or validation/).
# For a class-folder dataset, it should directly contain folders like Tomato___healthy/.
OUTPUT_DIR = Path("/kaggle/working/models") if Path("/kaggle/working").exists() else Path.cwd() / "models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset path:", DATA_ROOT)
print("Exists:", DATA_ROOT.exists())
print("Output path:", OUTPUT_DIR.resolve())
if not DATA_ROOT.exists():
    raise FileNotFoundError("Set DATA_ROOT to the real extracted dataset directory before continuing.")

Dataset path: \kaggle\input\new-plant-diseases-dataset\New Plant Diseases Dataset(Augmented)\New Plant Diseases Dataset(Augmented)
Exists: False
Output path: C:\Users\mmahe\Desktop\KBP\KrishiMitra\ML_Model\fast_plant_disease_bundle\fast_plant_disease_bundle\models


FileNotFoundError: Set DATA_ROOT to the real extracted dataset directory before continuing.

## 3. Detect the folder layout and load images

Class names are inferred from folder names. This notebook uses an existing `train/` and `valid/` split when present. Otherwise, it performs a stratified 80/20 split from class folders. If an independent `test/` folder containing one subfolder per class exists, it will be used for final evaluation.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def has_class_subfolders(folder: Path, min_classes=2):
    if not folder.exists() or not folder.is_dir():
        return False
    count = 0
    for sub in folder.iterdir():
        if sub.is_dir():
            try:
                if any(p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS for p in sub.iterdir()):
                    count += 1
            except PermissionError:
                pass
    return count >= min_classes

train_dir = DATA_ROOT / "train" if has_class_subfolders(DATA_ROOT / "train") else None
valid_dir = None
for name in ("valid", "validation", "val"):
    candidate = DATA_ROOT / name
    if has_class_subfolders(candidate):
        valid_dir = candidate
        break

test_candidate = DATA_ROOT / "test"
test_dir = test_candidate if has_class_subfolders(test_candidate) else None

if train_dir is not None:
    CLASS_LAYOUT = "split"
    print("Detected split layout.")
else:
    CLASS_LAYOUT = "single_root"
    if not has_class_subfolders(DATA_ROOT):
        raise FileNotFoundError(
            "Could not find class folders. DATA_ROOT must contain train/valid subfolders, "
            "or class folders directly (one folder per disease class)."
        )
    train_dir = DATA_ROOT
    print("Detected one-folder-per-class layout; will create a validation split.")

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.80, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])
val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

train_base = datasets.ImageFolder(train_dir, transform=train_transform)
class_names = train_base.classes
num_classes = len(class_names)
if num_classes < 2:
    raise ValueError(f"Found only {num_classes} class(es). Check DATA_ROOT.")

if CLASS_LAYOUT == "split" and valid_dir is not None:
    val_base = datasets.ImageFolder(valid_dir, transform=val_transform)
    if val_base.classes != class_names:
        raise ValueError("train and valid folders do not have matching class names. Check the dataset folders.")
    train_dataset = train_base
    val_dataset = val_base
else:
    # Separate transforms for train and validation; use the same file order and stratified indices.
    val_base = datasets.ImageFolder(train_dir, transform=val_transform)
    labels = np.array(train_base.targets)
    rng = np.random.default_rng(SEED)
    train_indices, val_indices = [], []
    for class_id in range(num_classes):
        class_indices = np.where(labels == class_id)[0]
        rng.shuffle(class_indices)
        n_val = max(1, int(0.20 * len(class_indices)))
        val_indices.extend(class_indices[:n_val].tolist())
        train_indices.extend(class_indices[n_val:].tolist())
    train_dataset = Subset(train_base, train_indices)
    val_dataset = Subset(val_base, val_indices)

# Optional stratified quick subset: smaller sample for a fast prototype.
def get_subset_labels(dataset):
    if isinstance(dataset, Subset):
        base = dataset.dataset
        return [base.targets[i] for i in dataset.indices]
    return list(dataset.targets)

def stratified_subset(dataset, fraction, seed=SEED):
    labels = np.array(get_subset_labels(dataset))
    rng = np.random.default_rng(seed)
    selected = []
    for class_id in np.unique(labels):
        class_positions = np.where(labels == class_id)[0]
        rng.shuffle(class_positions)
        take = max(1, int(round(len(class_positions) * fraction)))
        selected.extend(class_positions[:take].tolist())
    rng.shuffle(selected)
    return Subset(dataset, selected)

if QUICK_RUN and TRAIN_FRACTION < 1.0:
    train_dataset = stratified_subset(train_dataset, TRAIN_FRACTION)
    print(f"QUICK_RUN enabled: using {TRAIN_FRACTION:.0%} of training subset")

if test_dir is not None:
    test_dataset = datasets.ImageFolder(test_dir, transform=val_transform)
    if test_dataset.classes != class_names:
        print("Warning: test classes do not match training classes exactly; independent test evaluation will be skipped.")
        test_dataset = None
else:
    test_dataset = None

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
test_loader = (DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
               if test_dataset is not None else None)

print("Classes:", num_classes)
print("Images in train loader dataset:", len(train_dataset))
print("Images in validation dataset:", len(val_dataset))
print("Independent test set:", len(test_dataset) if test_dataset is not None else "not detected")
print("First class names:", class_names[:10])

## 4. View a small batch (optional)

In [ ]:
# Optional quick check. Uncomment if you want to inspect sample photos before training.
# imgs, labels = next(iter(train_loader))
# plt.figure(figsize=(10, 6))
# for i in range(min(8, len(imgs))):
#     plt.subplot(2, 4, i + 1)
#     img = imgs[i].permute(1, 2, 0).cpu().numpy()
#     mean = np.array([0.485, 0.456, 0.406])
#     std = np.array([0.229, 0.224, 0.225])
#     plt.imshow(np.clip(img * std + mean, 0, 1))
#     plt.title(class_names[int(labels[i])], fontsize=8)
#     plt.axis("off")
# plt.tight_layout()

## 5. Build MobileNetV3-Small with transfer learning

This uses ImageNet pretrained weights when available, then replaces the final classification layer with one output per disease class. The feature extractor is frozen initially, so the quick run trains the smaller classifier head rather than every layer from scratch.

In [ ]:
USE_PRETRAINED = True
try:
    weights = models.MobileNet_V3_Small_Weights.DEFAULT if USE_PRETRAINED else None
    model = models.mobilenet_v3_small(weights=weights)
    print("Loaded pretrained MobileNetV3-Small weights." if weights is not None else "Training from random initialization.")
except Exception as e:
    print("Could not load pretrained weights. Check Kaggle Internet setting. Falling back to random initialization.")
    print("Weights error:", repr(e))
    model = models.mobilenet_v3_small(weights=None)

for param in model.features.parameters():
    param.requires_grad = False

in_features = model.classifier[-1].in_features
model.classifier[-1] = nn.Linear(in_features, num_classes)
model = model.to(DEVICE)

print("Model:", "MobileNetV3-Small")
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("All parameters:", sum(p.numel() for p in model.parameters()))

## 6. Train quickly with GPU mixed precision when available

The quick configuration trains only the classification head for two epochs over 25% of the training examples. It is meant to produce a fast prototype, not a final quality guarantee. For the final model, set `QUICK_RUN = False` and consider increasing `EPOCHS` to 5–8, then rerun from this point.

In [ ]:
EPOCHS = 2
LEARNING_RATE = 1e-3
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=LEARNING_RATE, weight_decay=1e-4)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            outputs = model(images)
            loss = criterion(outputs, labels)
        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / max(total, 1), correct / max(total, 1)

history = {"train_loss": [], "val_loss": [], "val_accuracy": []}
best_val_acc = -1.0
best_state = copy.deepcopy(model.state_dict())

for epoch in range(EPOCHS):
    start = time.time()
    model.train()
    running_loss, seen = 0.0, 0
    for images, labels in train_loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * labels.size(0)
        seen += labels.size(0)

    train_loss = running_loss / max(seen, 1)
    val_loss, val_acc = evaluate(model, val_loader)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_acc)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())
    elapsed = time.time() - start
    print(f"Epoch {epoch + 1}/{EPOCHS} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.2%} | {elapsed/60:.1f} min")

model.load_state_dict(best_state)
print(f"Best validation accuracy from this run: {best_val_acc:.2%}")

## 7. Plot the training curves

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history["train_loss"], label="train loss")
plt.plot(history["val_loss"], label="validation loss")
plt.xlabel("Epoch")
plt.legend()
plt.title("Loss")
plt.subplot(1, 2, 2)
plt.plot(history["val_accuracy"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation accuracy")
plt.tight_layout()
plt.show()

## 8. Evaluate the held-out test folder, if available

In [ ]:
if test_loader is not None:
    test_loss, test_acc = evaluate(model, test_loader)
    print(f"Independent test loss: {test_loss:.4f}")
    print(f"Independent test accuracy: {test_acc:.2%}")
else:
    print("No independent test folder detected. Validation score was used for monitoring; do not call it test accuracy.")

## 9. Save the model and class labels for your project

The `.pth` checkpoint stores the model architecture name, learned weights, class mapping, and image size. Keep it together with `class_names.json`. This is a **PyTorch model**, not a TensorFlow `.keras` model.

In [ ]:
checkpoint_path = OUTPUT_DIR / "plant_disease_mobilenetv3_small.pth"
labels_path = OUTPUT_DIR / "class_names.json"

checkpoint = {
    "architecture": "mobilenet_v3_small",
    "num_classes": num_classes,
    "class_names": class_names,
    "image_size": IMAGE_SIZE,
    "model_state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
    "best_validation_accuracy": float(best_val_acc),
    "quick_run": bool(QUICK_RUN),
    "train_fraction": float(TRAIN_FRACTION if QUICK_RUN else 1.0),
}
torch.save(checkpoint, checkpoint_path)
with open(labels_path, "w", encoding="utf-8") as f:
    json.dump(class_names, f, indent=2, ensure_ascii=False)

print("Saved checkpoint:", checkpoint_path.resolve())
print("Saved labels:", labels_path.resolve())
print("For final training set QUICK_RUN=False and rerun from data loading/model setup onward.")

## 10. Test one image

Set `IMAGE_PATH` to a leaf photo you have not used for training. Predictions are probabilities over the known PlantVillage classes; a high score is not proof of a correct diagnosis.

In [ ]:
IMAGE_PATH = Path(r"/path/to/your/leaf_photo.jpg")  # CHANGE THIS

if IMAGE_PATH.exists():
    image = Image.open(IMAGE_PATH).convert("RGB")
    tensor = val_transform(image).unsqueeze(0).to(DEVICE)
    model.eval()
    with torch.no_grad():
        logits = model(tensor)
        probabilities = torch.softmax(logits, dim=1)[0]
    topk = torch.topk(probabilities, k=min(5, num_classes))
    plt.figure(figsize=(4, 4))
    plt.imshow(image)
    plt.axis("off")
    plt.show()
    for score, idx in zip(topk.values.cpu().tolist(), topk.indices.cpu().tolist()):
        print(f"{class_names[idx]}: {score:.2%}")
else:
    print("Edit IMAGE_PATH to an existing image file if you want to test single-image inference.")

## 11. Load the exported checkpoint in a separate Python application

Use the same MobileNetV3-Small architecture and image normalization during inference. Example loading code:

```python
import json
import torch
import torch.nn as nn
from torchvision import models

checkpoint = torch.load("models/plant_disease_mobilenetv3_small.pth", map_location="cpu")
model = models.mobilenet_v3_small(weights=None)
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, checkpoint["num_classes"])
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
class_names = checkpoint["class_names"]
```

For your inference pipeline, resize to 160×160 RGB and normalize with mean `(0.485, 0.456, 0.406)` and standard deviation `(0.229, 0.224, 0.225)`. These preprocessing steps must match the notebook. The exported model knows only the classes it was trained on.

### Time-saving tips
- Training on Kaggle GPU is generally much quicker than a laptop CPU.
- Keep `QUICK_RUN=True` while confirming your paths and dependencies; set it to `False` for the final model.
- Do not train every epoch from scratch if a saved checkpoint already exists; load the checkpoint and fine-tune instead.
- If the pretrained weights fail to download, enable Internet in Kaggle settings or choose a runtime with cached weights.